# PHASE 1.1 — Corrected

## Your Phase 1 run completed, and one of its numbers was wrong

The run finished cleanly: 58 minutes, 174 conditions, all fifteen ImageNet-C families valid, no
errors. Concern 6 was answered decisively. But the variance decomposition used the **wrong
bootstrap**, and the error ran in the direction that flatters this paper's thesis.

`analyse()` estimated the image-level standard error by resampling **rows**. The manuscript
pipeline resamples **images**. Each image contributes 30 rows, so a row bootstrap treats them as
independent, underestimates the standard error, and --- because
$\sigma_e^2 = \mathrm{SE}_{\rm image}^2\, m F$ --- **inflates** $\rho$.

Simulated at a known ICC of 0.080 with a realistic image effect, the row bootstrap returned a
standard error 3.4$\times$ too small and recovered $\rho = 0.059$ where the clustered bootstrap
recovered $0.004$. The direction is what matters. A bug that made the finding look weaker would
have been caught by disappointment; this one made it look stronger.

The Phase 1 values are therefore **not usable**: they gave $\rho$ in 0.099--0.311 where the
manuscript's clustered pipeline gives 0.029--0.306 for the same quantity. A reviewer comparing the
two would find them inconsistent, and the newer, wrong one is the flattering one.

## What survives unchanged

**Concern 6 does not depend on the decomposition** --- it rests on balanced accuracy under
corruption:

| Backbone | SDI-C | ImageNet-C |
|---|---|---|
| DINOv2 ViT-S/14 | **0.849** | **0.761** |
| ViT-B/16 | 0.833 | 0.709 |
| ConvNeXt-T | 0.818 | 0.690 |
| EfficientNet-B0 | 0.761 | 0.675 |
| ResNet-50 | 0.738 | 0.604 |

Kendall $\tau = +1.000$ ($p = 0.017$, the minimum attainable at $n=5$), the same winner under
both suites, and the cost of choosing on ImageNet-C measured on SDI-C is **exactly zero**. For
ranking backbones, SDI-C is redundant, and Section 3 has to say so.

**One finding got stronger.** ImageNet-C carried roughly 2.5$\times$ the family-level
heterogeneity of SDI-C ($\rho$ medians 0.32 versus 0.13). Both came from the biased estimator, so
the levels are unusable --- but both were biased the same way, so the ratio is informative. If it
survives correction, the overstatement this paper documents is *larger* for the benchmark the
field actually uses, which moves the central claim from a note about our own suite to a statement
about ImageNet-C.

## What changed here

The bootstrap is image-clustered, matching the manuscript exactly. The suite ICC comparison is
promoted from a diagnostic print to a reported result. And **features are persisted to disk**
(~1.7 GB): the first version cached in memory only, so an analysis bug cost a full re-extraction.

Re-extraction costs 58 minutes once more --- the features went with the runtime --- but after this
run, re-running the analysis is free.

## 0. Core module

In [ ]:
CORE_V2 = r"""
import math, hashlib
import numpy as np, cv2

def _clip(x): return np.clip(x, 0, 1)
def _mask3(mask, x): return mask[..., None] if x.ndim == 3 else mask

# ------------------------------------------------------------------ optics --
def defocus_blur(x, s):
    r = [1, 2, 3, 5, 7][s-1]
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r+1, 2*r+1)).astype(np.float32)
    return _clip(cv2.filter2D(x, -1, k/k.sum()))

def motion_blur(x, s):
    ksz = [5, 9, 13, 19, 25][s-1]
    ang = np.random.uniform(0, 180)
    k = np.zeros((ksz, ksz), np.float32); k[ksz//2, :] = 1.0
    M = cv2.getRotationMatrix2D((ksz/2-.5, ksz/2-.5), ang, 1.0)
    k = cv2.warpAffine(k, M, (ksz, ksz))
    return _clip(cv2.filter2D(x, -1, k/k.sum()))

def vignetting(x, s):
    st = [0.15, 0.30, 0.45, 0.62, 0.80][s-1]
    h, w = x.shape[:2]; yy, xx = np.mgrid[0:h, 0:w]
    r = np.sqrt(((xx-w/2)/(w/2))**2 + ((yy-h/2)/(h/2))**2)
    m = np.clip(1 - st*np.clip(r-0.4, 0, None)/0.6, 0, 1).astype(np.float32)
    return _clip(x * _mask3(m, x))

def lens_contamination(x, s):
    n = [3, 7, 13, 22, 34][s-1]; h, w = x.shape[:2]
    blurred = cv2.GaussianBlur(x, (0, 0), sigmaX=max(1.0, min(h, w)/40))
    mask = np.zeros((h, w), np.float32)
    for _ in range(n):
        c = (np.random.randint(0, w), np.random.randint(0, h))
        rad = np.random.randint(max(2, int(0.015*w)), max(4, int(0.07*w)))
        cv2.circle(mask, c, rad, 1.0, -1, lineType=cv2.LINE_AA)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(1.0, min(h, w)/60))
    m = _mask3(mask, x)                       # guard: 2-D input used to broadcast to (H,W,W)
    return _clip((x*(1-m) + blurred*m) * (1 - 0.25*m))

def vibration_jitter(x, s):
    amp = [0.6, 1.4, 2.6, 4.2, 6.5][s-1]; h, w = x.shape[:2]
    dx, dy = np.random.uniform(-amp, amp, 2)
    out = cv2.warpAffine(x, np.float32([[1, 0, dx], [0, 1, dy]]), (w, h),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    ksz = int(max(3, 2*round(amp)+1))
    k = np.zeros((ksz, ksz), np.float32); k[ksz//2, :] = 1.0
    M2 = cv2.getRotationMatrix2D((ksz/2-.5, ksz/2-.5), math.degrees(math.atan2(dy, dx)), 1.0)
    k = cv2.warpAffine(k, M2, (ksz, ksz))
    return _clip(cv2.filter2D(out, -1, k/max(k.sum(), 1e-8)))

# ------------------------------------------------------------------ sensor --
def gaussian_noise(x, s):
    return _clip(x + np.random.normal(0, [0.03, 0.06, 0.10, 0.16, 0.24][s-1], x.shape))

def shot_noise(x, s):
    lam = [80, 35, 15, 7, 3][s-1]
    return _clip(np.random.poisson(x*lam)/float(lam))

def scanline_banding(x, s):
    amp = [0.03, 0.06, 0.11, 0.17, 0.25][s-1]; h = x.shape[0]
    period = np.random.uniform(3, 22); phase = np.random.uniform(0, 2*np.pi)
    b = (1 + amp*np.sin(2*np.pi*np.arange(h)/period + phase)).astype(np.float32)
    return _clip(x * (b[:, None, None] if x.ndim == 3 else b[:, None]))

# ------------------------------------------------------------- photometric --
def illumination_gradient(x, s):
    st = [0.10, 0.20, 0.32, 0.46, 0.62][s-1]; h, w = x.shape[:2]
    ang = np.random.uniform(0, 2*np.pi); yy, xx = np.mgrid[0:h, 0:w]
    u = (xx/w-.5)*np.cos(ang) + (yy/h-.5)*np.sin(ang)
    g = (1 + st*u/(np.abs(u).max()+1e-8)).astype(np.float32)
    return _clip(x * _mask3(g, x))

def brightness_drift(x, s):
    # v2: gamma instead of an additive offset. Gamma maps [0,1] -> [0,1] and CANNOT clip.
    # v1 saturated 36.8% of pixels at severity 5 on Magnetic Tile, which destroyed dynamic
    # range rather than shifting brightness and made the top of the ladder meaningless.
    g = [1.12, 1.26, 1.45, 1.72, 2.05][s-1]
    if np.random.rand() < 0.5: g = 1.0/g
    return _clip(np.power(_clip(x), g))

def contrast_loss(x, s):
    g = [0.80, 0.65, 0.50, 0.36, 0.24][s-1]
    m = x.mean(axis=(0, 1), keepdims=True)
    return _clip((x-m)*g + m)

# ---------------------------------------------------------------- pipeline --
def jpeg_compression(x, s):
    q = [70, 50, 32, 18, 9][s-1]
    src = (x[..., ::-1]*255).astype(np.uint8) if x.ndim == 3 else (x*255).astype(np.uint8)
    _, enc = cv2.imencode(".jpg", src, [int(cv2.IMWRITE_JPEG_QUALITY), q])
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR if x.ndim == 3 else cv2.IMREAD_GRAYSCALE)
    return (dec[..., ::-1] if x.ndim == 3 else dec).astype(np.float32)/255.

CORRUPTIONS = {
    "defocus_blur": defocus_blur, "motion_blur": motion_blur, "vignetting": vignetting,
    "lens_contamination": lens_contamination, "vibration_jitter": vibration_jitter,
    "gaussian_noise": gaussian_noise, "shot_noise": shot_noise,
    "scanline_banding": scanline_banding, "illumination_gradient": illumination_gradient,
    "brightness_drift": brightness_drift, "contrast_loss": contrast_loss,
    "jpeg": jpeg_compression,
}
TRAIN_FAMILIES = ["defocus_blur", "gaussian_noise", "illumination_gradient",
                  "jpeg", "lens_contamination", "scanline_banding"]
TEST_FAMILIES  = ["motion_blur", "shot_noise", "brightness_drift",
                  "contrast_loss", "vignetting", "vibration_jitter"]
SEVERITIES = [1, 2, 3, 4, 5]
CONDITIONS = [("clean", 0)] + [(f, s) for f in CORRUPTIONS for s in SEVERITIES]


def corruption_seed(image_id, family, severity=None):
    # Depends on (image_id, family) only, NOT severity: nuisance parameters (blur angle,
    # banding period, blob positions, gradient orientation) stay fixed so that severity is
    # the sole varying factor. Seeding on severity too made scanline_banding score 0.20.
    h = hashlib.sha256(f"{image_id}|{family}".encode()).digest()
    return int.from_bytes(h[:4], "little")


def apply_corruption(img_u8, family, severity, image_id=None, seed=None):
    if family == "clean":
        return img_u8
    if seed is None and image_id is not None:
        seed = corruption_seed(image_id, family)
    st = None
    if seed is not None:
        st = np.random.get_state(); np.random.seed(seed % (2**32))
    try:
        out = CORRUPTIONS[family](img_u8.astype(np.float32)/255., severity)
    finally:
        if st is not None: np.random.set_state(st)
    return (np.clip(out, 0, 1)*255).astype(np.uint8)


# ------------------------------------------------------- quality descriptors -
def _gray(im):
    g = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY) if im.ndim == 3 else im
    return g.astype(np.float32)/255.

def _sharp(im): return float(cv2.Laplacian(_gray(im), cv2.CV_32F).var())

def _noise(im):
    g = _gray(im); h, w = g.shape
    M = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    return float(np.abs(cv2.filter2D(g, -1, M)).sum()*math.sqrt(math.pi/2) /
                 (6*max(w-2, 1)*max(h-2, 1)))

def _block(im):
    dh = np.abs(np.diff(_gray(im), axis=1))
    on = dh[:, 7::8].mean() if dh.shape[1] > 8 else 0.0
    return float(on/(dh.mean()+1e-8) - 1.0)

def _hf(im):
    g = _gray(im); f = np.abs(np.fft.fftshift(np.fft.fft2(g))); h, w = g.shape
    cy, cx = h//2, w//2; r = max(4, min(h, w)//8)
    return float(1.0 - f[cy-r:cy+r, cx-r:cx+r].sum()/(f.sum()+1e-8))


def quality_descriptor(img_u8):
    # 8-d ABSOLUTE no-reference descriptor (unchanged from v1).
    g = _gray(img_u8); h, w = g.shape
    if img_u8.ndim == 3:
        rg = img_u8[..., 0].astype(np.float32) - img_u8[..., 1]
        yb = .5*(img_u8[..., 0].astype(np.float32) + img_u8[..., 1]) - img_u8[..., 2]
        colour = float((np.sqrt(rg.std()**2 + yb.std()**2)
                        + .3*np.sqrt(rg.mean()**2 + yb.mean()**2))/255.)
    else:
        colour = 0.0
    cen = g[h//4:3*h//4, w//4:3*w//4]
    per = (g.sum()-cen.sum())/max(g.size-cen.size, 1)
    v = np.array([math.log1p(max(_sharp(img_u8), 0.)*1e3), _hf(img_u8),
                  math.log1p(max(_noise(img_u8), 0.)*1e3), _block(img_u8),
                  float(g.mean()), float(g.std()), colour,
                  float(cen.mean()/(per+1e-8))], np.float32)
    return np.nan_to_num(v, nan=0., posinf=0., neginf=0.)


def quality_descriptor_relative(img_u8):
    # 8-d PERTURBATION-RESPONSE descriptor: how much does a statistic move when a known
    # perturbation is applied? Higher severity signal than the absolute set (0.324 vs 0.236
    # on the synthetic benchmark) but NO reduction in class leakage on its own -- ratios
    # remove the absolute texture level, not the spectral shape. Use with class-conditional
    # standardisation, never alone.
    g8 = (_gray(img_u8)*255).astype(np.uint8); eps = 1e-8
    b = cv2.GaussianBlur(g8, (0, 0), 1.5)
    dn = cv2.resize(cv2.resize(g8, (max(2, g8.shape[1]//2), max(2, g8.shape[0]//2)),
                               interpolation=cv2.INTER_AREA),
                    (g8.shape[1], g8.shape[0]), interpolation=cv2.INTER_LINEAR)
    rn = np.random.default_rng(12345)
    nz = np.clip(g8.astype(np.float32) + rn.normal(0, 12, g8.shape), 0, 255).astype(np.uint8)
    _, e = cv2.imencode(".jpg", g8, [int(cv2.IMWRITE_JPEG_QUALITY), 40])
    jp = cv2.imdecode(e, cv2.IMREAD_GRAYSCALE)
    kh = np.zeros((9, 9), np.float32); kh[4, :] = 1/9.
    hb = cv2.filter2D(g8.astype(np.float32), -1, kh).astype(np.uint8)
    vb = cv2.filter2D(g8.astype(np.float32), -1, kh.T).astype(np.uint8)
    x = g8.astype(np.float32)/255.
    v = np.array([
        math.log((_sharp(g8)+eps)/(_sharp(b)+eps)),
        math.log((_sharp(g8)+eps)/(_sharp(dn)+eps)),
        math.log((_noise(nz)+eps)/(_noise(g8)+eps)),
        _block(jp) - _block(g8),
        float(((x+0.25) > 1.0).mean() + ((x-0.25) < 0.0).mean()),
        abs(math.log((_sharp(hb)+eps)/(_sharp(vb)+eps))),
        math.log((_hf(g8)+eps)/(_hf(b)+eps)),
        math.log((_gray(g8).std()+eps)/(_gray(b).std()+eps)),
    ], np.float32)
    return np.nan_to_num(v, nan=0., posinf=0., neginf=0.)


QUALITY_NAMES = ["sharpness", "hf_energy", "noise", "blockiness",
                 "luminance_mean", "luminance_std", "colourfulness", "vignette_ratio"]
RELATIVE_NAMES = ["blur_headroom", "resolution_headroom", "noise_headroom",
                  "compression_headroom", "clipping_headroom", "blur_anisotropy",
                  "hf_retention", "contrast_retention"]
QUALITY_DIM = 8


class ClassConditionalStandardiser:
    # z = (q - mu_c) / sigma_c, with mu_c and sigma_c estimated on DEVELOPMENT data only.
    #
    # Rationale: degradation is relative. A blurry-looking crazing image and a sharp-looking
    # patches image can have identical absolute sharpness; what makes one degraded is that it
    # is blurrier than crazing images normally are. Removing the class-conditional mean strips
    # the content component and leaves the deviation-from-typical -- which is the degradation.
    #
    # At test time c is the model's own prediction. There is no feedback loop: a per-image
    # scalar temperature cannot change the argmax, so the prediction is fixed before the
    # calibrator runs.
    def __init__(self, n_classes):
        self.n_classes = n_classes; self.mu = None; self.sd = None

    def fit(self, q, y):
        d = q.shape[1]
        self.mu = np.zeros((self.n_classes, d), np.float32)
        self.sd = np.ones((self.n_classes, d), np.float32)
        gm, gs = q.mean(0), q.std(0) + 1e-6
        for c in range(self.n_classes):
            m = (y == c)
            if m.sum() >= 5:                 # fall back to global stats for tiny classes
                self.mu[c] = q[m].mean(0); self.sd[c] = q[m].std(0) + 1e-6
            else:
                self.mu[c] = gm; self.sd[c] = gs
        return self

    def transform(self, q, y_pred):
        y_pred = np.asarray(y_pred).astype(int)
        return ((q - self.mu[y_pred]) / self.sd[y_pred]).astype(np.float32)
"""
print(f"embedded core module: {len(CORE_V2.splitlines())} lines")

embedded core module: 246 lines


## 1. Setup, and repairing the ImageNet-C reference implementation

In [ ]:
#@title Dependencies
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","timm==1.0.11","scikit-learn==1.5.2",
                "opencv-python-headless==4.10.0.84","pandas==2.2.3","kagglehub","tqdm",
                "imagecorruptions==1.1.2","setuptools"],check=True)
print("ok")

ok


In [ ]:
import os, json, math, time, random, hashlib, warnings
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import timm
warnings.filterwarnings("ignore")

SEED=20260821
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
ROOT=Path("/content/sdic"); OUT=ROOT/"phase1"; OUT.mkdir(parents=True,exist_ok=True)
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

sys.path.insert(0,str(ROOT))
_t=ROOT/"sdic_core_v2.py"; _w=hashlib.sha256(CORE_V2.encode()).hexdigest()
if not _t.exists() or hashlib.sha256(_t.read_text().encode()).hexdigest()!=_w:
    _t.write_text(CORE_V2)
import importlib, sdic_core_v2; importlib.reload(sdic_core_v2)
from sdic_core_v2 import (CORRUPTIONS, TRAIN_FAMILIES, TEST_FAMILIES, SEVERITIES,
                          apply_corruption, corruption_seed, quality_descriptor, QUALITY_DIM)
print("device:", DEVICE, "| sdic_core_v2", _w[:12])

device: cuda | sdic_core_v2 fd8e3c534562


In [ ]:
#@title Repair and validate the ImageNet-C reference implementation
# 1) scikit-image >= 0.19 removed the `multichannel` keyword -> glass_blur raises TypeError
if not hasattr(np, "float_"):
    np.float_ = np.float64            # 2) NumPy 2.0 removed np.float_ -> fog raises AttributeError
import skimage.filters as _skf
_orig_gaussian = _skf.gaussian
def _gaussian(image, *a, **kw):
    if "multichannel" in kw:
        kw.setdefault("channel_axis", -1 if kw.pop("multichannel") else None)
    return _orig_gaussian(image, *a, **kw)
_skf.gaussian = _gaussian
import imagecorruptions.corruptions as _icc
_icc.gaussian = _gaussian
from imagecorruptions import corrupt as _ic_corrupt, get_corruption_names

INC_FAMILIES = list(get_corruption_names())

def apply_inc(img_u8, family, severity, image_id):
    """ImageNet-C corruption, seeded per (image, family) exactly as SDI-C is, so that
    severity is the only factor that varies and the suite is reproducible."""
    st = np.random.get_state()
    np.random.seed(corruption_seed(image_id, family) % (2**32))
    try:
        out = _ic_corrupt(img_u8, corruption_name=family, severity=severity)
    finally:
        np.random.set_state(st)
    return np.ascontiguousarray(out, dtype=np.uint8)

# assert every family works BEFORE anything expensive runs
_probe = (np.random.default_rng(0).integers(30,220,(200,200,3))).astype(np.uint8)
_bad=[]
for f in INC_FAMILIES:
    try:
        o=apply_inc(_probe,f,3,"probe")
        assert o.shape==_probe.shape and o.dtype==np.uint8 and np.isfinite(o).all()
    except Exception as e:
        _bad.append((f,f"{type(e).__name__}: {e}"))
assert not _bad, f"ImageNet-C families still broken: {_bad}"
a=apply_inc(_probe,"gaussian_noise",3,"x"); b=apply_inc(_probe,"gaussian_noise",3,"x")
assert np.array_equal(a,b), "ImageNet-C not reproducible under our seeding"
print(f"all {len(INC_FAMILIES)} ImageNet-C families valid and reproducible")
print(" ", INC_FAMILIES)

all 15 ImageNet-C families valid and reproducible
  ['gaussian_noise', 'shot_noise', 'impulse_noise', 'defocus_blur', 'glass_blur', 'motion_blur', 'zoom_blur', 'snow', 'frost', 'fog', 'brightness', 'contrast', 'elastic_transform', 'pixelate', 'jpeg_compression']


## 2. Data

NEU-CLS is the shared testbed. Magnetic Tile and KolektorSDD2 are wired in but disabled by
default: MT still needs folds rebuilt without the discarded pHash grouping, and KSDD2 needs the
official ViCoS release. Enabling either only requires flipping the switch once those are done.

In [ ]:
#@title Index NEU-CLS
import kagglehub
NEU_ROOT=Path(kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database"))
neu=[p for p in sorted(NEU_ROOT.rglob("*.jpg"))
     if p.parent.name.lower() not in {"train","validation","images","annotations"}]
labels=np.array([p.parent.name for p in neu])
LUT={n:i for i,n in enumerate(sorted(set(labels)))}
y_all=np.array([LUT[l] for l in labels]); NC=len(LUT)
def rd(p): return cv2.cvtColor(cv2.imread(str(p)),cv2.COLOR_BGR2RGB)
assert len(neu)==1800 and NC==6, f"unexpected NEU index: {len(neu)} images, {NC} classes"

N_PER_CLASS   = 150   #@param {type:"integer"}
N_HEADTOHEAD  = 50    #@param {type:"integer"}
set_seed()
per=defaultdict(list)
for p,yy in zip(neu,y_all): per[yy].append(p)
sel=[]
for c,v in per.items():
    pick=np.random.default_rng(SEED+c).choice(len(v),min(N_PER_CLASS,len(v)),replace=False)
    sel += [neu.index(v[t]) for t in pick]
sel=np.array(sorted(sel)); S_paths=[neu[i] for i in sel]; S_y=y_all[sel]

# paired subset for the suite head-to-head (same images for both suites)
h2h=[]
for c in range(NC):
    idx=np.where(S_y==c)[0]
    h2h += list(np.random.default_rng(SEED+100+c).choice(idx,min(N_HEADTOHEAD,len(idx)),replace=False))
H2H=np.array(sorted(h2h))
print(f"main set {len(S_paths)} images | head-to-head subset {len(H2H)} images "
      f"({dict(Counter(S_y[H2H]))})")

Using Colab cache for faster access to the 'neu-surface-defect-database' dataset.
main set 900 images | head-to-head subset 300 images ({np.int64(0): 50, np.int64(1): 50, np.int64(2): 50, np.int64(3): 50, np.int64(4): 50, np.int64(5): 50})


## 3. Extraction — conditions outer, backbones inner

Each corrupted image is generated once and pushed through all five backbones, using each
checkpoint's own data configuration.

In [ ]:
from timm.data import resolve_model_data_config
_INTERP={"bilinear":cv2.INTER_LINEAR,"bicubic":cv2.INTER_CUBIC,
         "nearest":cv2.INTER_NEAREST,"area":cv2.INTER_AREA}
BACKBONES=["resnet50.a1_in1k","tf_efficientnet_b0.ns_jft_in1k","convnext_tiny.fb_in22k_ft_in1k",
           "vit_base_patch16_224.augreg2_in21k_ft_in1k","vit_small_patch14_reg4_dinov2.lvd142m"]
SHORT={b:b.split(".")[0] for b in BACKBONES}

MODELS, PP = {}, {}
for b in BACKBONES:
    try:
        m=timm.create_model(b,pretrained=True,num_classes=0,img_size=224).eval().to(DEVICE)
    except TypeError:
        m=timm.create_model(b,pretrained=True,num_classes=0).eval().to(DEVICE)
    c=resolve_model_data_config(m)
    MODELS[b]=m
    PP[b]={"mean":np.array(c["mean"],np.float32),"std":np.array(c["std"],np.float32),"size":224,
           "interp":_INTERP.get(c["interpolation"],cv2.INTER_CUBIC),
           "crop_pct":float(c.get("crop_pct") or 1.0)}
    print(f"  {SHORT[b]:28s} mean={np.round(c['mean'],3).tolist()} crop={c.get('crop_pct')} "
          f"interp={c['interpolation']}")

def prep(im,b):
    q=PP[b]; sz=q["size"]; to=int(round(sz/q["crop_pct"]))
    h,w=im.shape[:2]; s=to/min(h,w)
    r=cv2.resize(im,(max(1,int(round(w*s))),max(1,int(round(h*s)))),interpolation=q["interp"])
    hh,ww=r.shape[:2]; t,l=(hh-sz)//2,(ww-sz)//2
    x=(r[t:t+sz,l:l+sz].astype(np.float32)/255.-q["mean"])/q["std"]
    return torch.from_numpy(x).permute(2,0,1)

@torch.no_grad()
def extract_condition(paths, ids, suite, family, severity, bs=64):
    """Generate the corrupted batch ONCE, push it through every backbone."""
    feats={b:[] for b in BACKBONES}; qs=[]
    for i in range(0,len(paths),bs):
        chunk=list(zip(paths[i:i+bs], ids[i:i+bs]))
        imgs=[]
        for p,iid in chunk:
            im=rd(p)
            if family!="clean":
                im=(apply_corruption(im,family,severity,image_id=iid) if suite=="sdic"
                    else apply_inc(im,family,severity,iid))
            qs.append(quality_descriptor(im)); imgs.append(im)
        for b in BACKBONES:
            x=torch.stack([prep(im,b) for im in imgs]).to(DEVICE)
            with torch.autocast("cuda",enabled=DEVICE=="cuda"):
                feats[b].append(MODELS[b](x).float().cpu().numpy())
    return ({b:np.concatenate(v) for b,v in feats.items()},
            np.stack(qs[:len(paths)]) if len(qs)>=len(paths) else np.stack(qs))

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

  resnet50                     mean=[0.485, 0.456, 0.406] crop=0.95 interp=bicubic


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

  vit_base_patch16_224         mean=[0.5, 0.5, 0.5] crop=0.9 interp=bicubic


model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

  vit_small_patch14_reg4_dinov2 mean=[0.485, 0.456, 0.406] crop=1.0 interp=bicubic


In [ ]:
#@title Run extraction  (the long cell)
SDIC_CONDS=[("clean",0)]+[(f,s) for f in TRAIN_FAMILIES for s in (1,3,5)] \
                        +[(f,s) for f in TEST_FAMILIES for s in SEVERITIES]
INC_CONDS =[("clean",0)]+[(f,s) for f in INC_FAMILIES for s in SEVERITIES]

FEAT=ROOT/"feats_phase1"; FEAT.mkdir(parents=True,exist_ok=True)
CACHE={}          # (suite, scope, family, sev) -> (dict backbone->feats, q)

def _fp(suite,scope,fam,sev): return FEAT/f"{suite}__{scope}__{fam}__{sev}.npz"

def _load(k):
    p=_fp(*k)
    if not p.exists(): return None
    z=np.load(p)
    return ({b:z[f"f_{SHORT[b]}"] for b in BACKBONES}, z["q"])

def _save(k,val):
    feats,q=val
    np.savez_compressed(_fp(*k), q=q, **{f"f_{SHORT[b]}":feats[b] for b in BACKBONES})

def run(scope, idx, suite, conds):
    # Extraction is persisted, so re-running the analysis costs nothing. The first version of
    # this notebook cached in memory only; an analysis bug then cost a full 58-minute redo.
    paths=[S_paths[i] for i in idx]; ids=[S_paths[i].stem for i in idx]
    hit=0
    for fam,sev in tqdm(conds, desc=f"{suite}/{scope}"):
        k=(suite,scope,fam,sev)
        if k in CACHE: continue
        cached=_load(k)
        if cached is not None:
            CACHE[k]=cached; hit+=1; continue
        CACHE[k]=extract_condition(paths,ids,suite,fam,sev); _save(k,CACHE[k])
    if hit: print(f"    {hit}/{len(conds)} conditions loaded from disk")

t0=time.time()
run("main", np.arange(len(S_paths)), "sdic", SDIC_CONDS)          # concern 3
print(f"  SDI-C main done at {(time.time()-t0)/60:.1f} min")
run("h2h", H2H, "sdic", SDIC_CONDS)                               # concern 6, paired
run("h2h", H2H, "inc",  INC_CONDS)
for m in MODELS.values(): del m
MODELS.clear(); torch.cuda.empty_cache()
sz=sum(p.stat().st_size for p in FEAT.glob("*.npz"))/1e9
print(f"\nextraction complete in {(time.time()-t0)/60:.1f} min | {len(CACHE)} conditions"
      f" | {sz:.2f} GB cached at {FEAT}")
print("Re-running the analysis cells is now free; the features are on disk.")

sdic/main:   0%|          | 0/49 [00:00<?, ?it/s]

  SDI-C main done at 19.5 min


sdic/h2h:   0%|          | 0/49 [00:00<?, ?it/s]

inc/h2h:   0%|          | 0/76 [00:00<?, ?it/s]


extraction complete in 66.3 min | 174 conditions | 1.02 GB cached at /content/sdic/feats_phase1
Re-running the analysis cells is now free; the features are on disk.


## 4. Analysis machinery

Identical to Phase 0.11 so the numbers stay comparable: frozen probe on clean features, eleven
calibration arms, five folds, and the variance-components decomposition.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, train_test_split
from scipy.stats import f as fdist, kendalltau, t as tdist

EPS=1e-2
class ScalarT(nn.Module):
    def __init__(self,d=None):
        super().__init__(); self.log_t=nn.Parameter(torch.zeros(()))
    def temperature(self,q): return self.log_t.exp().expand(q.shape[0])+EPS
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)

class LinearT(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.register_buffer("mu",torch.zeros(d)); self.register_buffer("sd",torch.ones(d))
        self.lin=nn.Linear(d,1); nn.init.zeros_(self.lin.weight)
        nn.init.constant_(self.lin.bias,math.log(math.exp(1.0-EPS)-1.0))
    def fit_norm(self,q): self.mu.copy_(q.mean(0)); self.sd.copy_(q.std(0).clamp_min(1e-6))
    def temperature(self,q): return F.softplus(self.lin((q-self.mu)/self.sd).squeeze(-1))+EPS
    def forward(self,lg,q): return lg/self.temperature(q).unsqueeze(-1)

def fit_cal(cls,L,Q,Y,epochs=400,lr=1e-2):
    set_seed()
    L=torch.as_tensor(L,dtype=torch.float32); Q=torch.as_tensor(Q,dtype=torch.float32)
    Y=torch.as_tensor(Y,dtype=torch.long)
    m=cls(Q.shape[1])
    if hasattr(m,"fit_norm"): m.fit_norm(Q)
    opt=torch.optim.Adam(m.parameters(),lr=lr)
    for _ in range(epochs):
        opt.zero_grad(); F.cross_entropy(m(L,Q),Y).backward(); opt.step()
    return m.eval()

def softmax(z):
    z=z-z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def nll_pi(p,y): return -np.log(np.clip(p[np.arange(len(y)),y],1e-12,None))
def bacc(pred,y):
    from sklearn.metrics import balanced_accuracy_score
    return float(balanced_accuracy_score(y,pred))

def variance_components(per_family_deltas, se_image, n_img, n_fam):
    """Return (sigma_a, sigma_e, rho, CI_lo, CI_hi, SE ratio) for model d_if = mu + a_f + e_if."""
    d=np.asarray(per_family_deltas,float)
    var_fm=d.var(ddof=1); var_e=se_image**2*(n_img*n_fam)
    var_a=max(var_fm-var_e/n_img,0.0)
    rho=var_a/(var_a+var_e) if (var_a+var_e)>0 else 0.0
    Fst=1+n_img*var_a/var_e if var_e>0 else np.inf
    FL=fdist.ppf(.975,n_fam-1,n_fam*(n_img-1)); FU=fdist.ppf(.975,n_fam*(n_img-1),n_fam-1)
    fl,fu=Fst/FL,Fst*FU
    lo,hi=(fl-1)/(fl+n_img-1),(fu-1)/(fu+n_img-1)
    return (math.sqrt(var_a),math.sqrt(var_e),rho,max(lo,0.0),min(hi,1.0),
            math.sqrt(var_fm/n_fam)/se_image)

In [ ]:
def analyse(suite, scope, test_families, idx, n_folds=5, n_boot=4000):
    """One (suite, scope) -> per-backbone family-level statistics for C4 vs C2 and C4 vs C5."""
    yv=S_y[idx]; n=len(idx)
    cal_conds=[("clean",0)]+[(f,s) for f in TRAIN_FAMILIES for s in (1,3,5)] \
              if suite=="sdic" else [("clean",0)]+[(f,s) for f in INC_FAMILIES[:7] for s in (1,3,5)]
    tst_families=test_families
    rows=[]
    skf=StratifiedKFold(n_folds,shuffle=True,random_state=SEED)
    FOLDS=[te for _,te in skf.split(np.zeros(n),yv)]

    for b in BACKBONES:
        Fc,Qc=CACHE[(suite,scope,"clean",0)][0][b],CACHE[(suite,scope,"clean",0)][1]
        POOLF=defaultdict(lambda: defaultdict(list)); POOL=defaultdict(lambda: defaultdict(list))
        for k in range(n_folds):
            te=FOLDS[k]; rest=np.concatenate([FOLDS[j] for j in range(n_folds) if j!=k])
            tr,va=train_test_split(rest,test_size=0.25,random_state=SEED+k,stratify=yv[rest])
            pr=make_pipeline(StandardScaler(),
                             LogisticRegression(max_iter=4000,class_weight="balanced"))
            pr.fit(Fc[tr],yv[tr])
            lg=lambda Fx: (pr.decision_function(Fx) if pr.decision_function(Fx).ndim>1
                           else np.stack([-pr.decision_function(Fx)]*2,1))
            def gather(ix,conds):
                L,Q,Y=[],[],[]
                for fam,sev in conds:
                    Fx,Qx=CACHE[(suite,scope,fam,sev)][0][b],CACHE[(suite,scope,fam,sev)][1]
                    L.append(lg(Fx[ix])); Q.append(Qx[ix]); Y.append(yv[ix])
                L=np.concatenate(L); Q=np.concatenate(Q); Y=np.concatenate(Y)
                H=np.zeros((len(L),NC),np.float32); H[np.arange(len(L)),L.argmax(1)]=1.
                return L,Q,H,Y
            Lf,Qf,Hf,Yf=gather(va,cal_conds)
            arms={"C2":(fit_cal(ScalarT,Lf,Qf,Yf),"q"),
                  "C4":(fit_cal(LinearT,Lf,Qf,Yf),"q"),
                  "C5":(fit_cal(LinearT,Lf,Hf,Yf),"h")}
            for fam in tst_families:
                Lt,Qt,Ht,Yt=gather(te,[(fam,s) for s in SEVERITIES])
                feats={"q":Qt,"h":Ht}
                for nm,(mdl,ft) in arms.items():
                    with torch.no_grad():
                        p=softmax(mdl(torch.as_tensor(Lt,dtype=torch.float32),
                                      torch.as_tensor(feats[ft],dtype=torch.float32)).numpy())
                    POOLF[nm][fam].append(nll_pi(p,Yt))
                    POOL[nm]["all"].append(nll_pi(p,Yt))
                    POOL[nm]["im"].append(np.tile(idx[te],len(SEVERITIES)))
                    POOLF[nm][("acc",fam)].append((p.argmax(1)==Yt).astype(float))
        for nm in POOLF:
            for k2 in POOLF[nm]: POOLF[nm][k2]=np.concatenate(POOLF[nm][k2])
            POOL[nm]["all"]=np.concatenate(POOL[nm]["all"])
            POOL[nm]["im"]=np.concatenate(POOL[nm]["im"])

        rng=np.random.default_rng(SEED)
        for A,B in [("C4","C2"),("C4","C5")]:
            d_all=POOL[A]["all"]-POOL[B]["all"]; im_all=POOL[A]["im"]
            # IMAGE-CLUSTERED, matching the manuscript. Each image contributes 30 rows;
            # resampling rows treats them as independent, underestimates SE_image, and --
            # since var_e = SE_image^2 * n_img * n_fam -- INFLATES rho. Verified in
            # simulation: at a true ICC of 0.080 the row bootstrap returned 0.059 and the
            # clustered one 0.004. The bias flatters this paper's thesis, so it must not stand.
            uniq=np.unique(im_all); by={u:np.where(im_all==u)[0] for u in uniq}
            bs=np.array([d_all[np.concatenate([by[u] for u in
                         rng.choice(uniq,len(uniq),True)])].mean() for _ in range(n_boot)])
            se_img=(np.quantile(bs,.975)-np.quantile(bs,.025))/(2*1.96)
            per_fam=[float(POOLF[A][f].mean()-POOLF[B][f].mean()) for f in tst_families]
            sa,se,rho,lo,hi,ratio=variance_components(per_fam,se_img,n,len(tst_families))
            rows.append(dict(suite=suite,scope=scope,backbone=SHORT[b],comparison=f"{A} vs {B}",
                             sigma_a=sa,sigma_e=se,rho=rho,rho_lo=lo,rho_hi=hi,
                             se_ratio=ratio,per_family=per_fam))
        # robustness of the backbone itself, for the suite ranking
        acc=np.mean([POOLF["C2"][("acc",f)].mean() for f in tst_families])
        rows.append(dict(suite=suite,scope=scope,backbone=SHORT[b],comparison="robust_acc",
                         sigma_a=np.nan,sigma_e=np.nan,rho=np.nan,rho_lo=np.nan,rho_hi=np.nan,
                         se_ratio=np.nan,per_family=[float(POOLF["C2"][("acc",f)].mean())
                                                     for f in tst_families],robust_acc=acc))
    return pd.DataFrame(rows)

## 5. Concern 3 — does the finding hold across backbones?

In [ ]:
R3=analyse("sdic","main",TEST_FAMILIES,np.arange(len(S_paths)))
R3.to_csv(OUT/"phase1_concern3.csv",index=False)
V=R3[R3.comparison!="robust_acc"]
disp=V[["backbone","comparison","sigma_a","sigma_e","rho","rho_lo","rho_hi","se_ratio"]].round(4)
display(disp)

print("\nCONCERN 3 — distribution across the five backbones")
for cmp_ in V.comparison.unique():
    g=V[V.comparison==cmp_]
    print(f"  {cmp_}:  rho median {g.rho.median():.3f}  range {g.rho.min():.3f}-{g.rho.max():.3f}"
          f" | SE ratio median {g.se_ratio.median():.1f}x range "
          f"{g.se_ratio.min():.1f}-{g.se_ratio.max():.1f}x")
allpos=(V.rho_lo>0).all()
print(f"\n  every rho interval excludes zero: {allpos}"
      f"  ({int((V.rho_lo>0).sum())}/{len(V)})")
print("  -> the manuscript can state the result as a property of the benchmark rather than"
      "\n     of one architecture, and report the across-backbone range." if allpos else
      "  -> at least one backbone shows no family effect; report that heterogeneity honestly.")

,backbone,comparison,sigma_a,sigma_e,rho,rho_lo,rho_hi,se_ratio
0,resnet50,C4 vs C2,0.0440,0.1612,0.0693,0.0275,0.3121,8.2482
1,resnet50,C4 vs C5,0.2198,1.1444,0.0356,0.0135,0.1853,5.8476
3,tf_efficientnet_b0,C4 vs C2,0.0598,0.1781,0.1014,0.0415,0.4064,10.1276
4,tf_efficientnet_b0,C4 vs C5,0.1919,0.7457,0.0621,0.0245,0.2878,7.7852
6,convnext_tiny,C4 vs C2,0.0231,0.1046,0.0467,0.0180,0.2308,6.7123
7,convnext_tiny,C4 vs C5,0.0520,0.2823,0.0328,0.0124,0.1734,5.6172
9,vit_base_patch16_224,C4 vs C2,0.0478,0.1504,0.0916,0.0371,0.3797,9.5768
10,vit_base_patch16_224,C4 vs C5,0.3386,1.8289,0.0331,0.0125,0.1748,5.6429
12,vit_small_patch14_reg4_dinov2,C4 vs C2,0.0741,0.2404,0.0868,0.0350,0.3660,9.3010
13,vit_small_patch14_reg4_dinov2,C4 vs C5,0.9700,3.4922,0.0716,0.0285,0.3196,8.3923



CONCERN 3 — distribution across the five backbones
  C4 vs C2:  rho median 0.087  range 0.047-0.101 | SE ratio median 9.3x range 6.7-10.1x
  C4 vs C5:  rho median 0.036  range 0.033-0.072 | SE ratio median 5.8x range 5.6-8.4x

  every rho interval excludes zero: True  (10/10)
  -> the manuscript can state the result as a property of the benchmark rather than
     of one architecture, and report the across-backbone range.


## 6. Concern 6 — does SDI-C tell you anything ImageNet-C does not?

The test that matters is not whether the suites produce different numbers — they trivially will.
It is whether they produce different **decisions**: would a practitioner choosing a backbone on
ImageNet-C pick the same one SDI-C picks, and what does the mistake cost?

In [ ]:
R6s=analyse("sdic","h2h",TEST_FAMILIES,H2H)
R6i=analyse("inc","h2h",INC_FAMILIES[7:],H2H)   # held-out half, mirroring the SDI-C protocol
R6=pd.concat([R6s,R6i],ignore_index=True)
R6.to_csv(OUT/"phase1_concern6.csv",index=False)

rank=(R6[R6.comparison=="robust_acc"]
      .pivot_table(index="backbone",columns="suite",values="robust_acc"))
rank["rank_sdic"]=rank["sdic"].rank(ascending=False)
rank["rank_inc"] =rank["inc"].rank(ascending=False)
display(rank.round(4))

tau=kendalltau(rank["sdic"],rank["inc"])
best_s=rank["sdic"].idxmax(); best_i=rank["inc"].idxmax()
cost=rank.loc[best_s,"sdic"]-rank.loc[best_i,"sdic"]
print(f"\nCONCERN 6 — ranking agreement between the suites")
print(f"  Kendall tau = {tau.statistic:+.3f}  (p = {tau.pvalue:.3f})")
print(f"  most robust on SDI-C       : {best_s}")
print(f"  most robust on ImageNet-C  : {best_i}")
print(f"  cost of choosing on ImageNet-C, measured on SDI-C: {cost:+.4f} balanced accuracy")

vs=R6[R6.comparison!="robust_acc"].groupby("suite")["rho"].agg(["median","min","max"])
display(vs.round(4))
si,ss=vs.loc["inc","median"],vs.loc["sdic","median"]
print(f"\nFAMILY-LEVEL HETEROGENEITY BY SUITE")
print(f"  ImageNet-C rho median {si:.3f} | SDI-C rho median {ss:.3f}"
      f"  -> ImageNet-C {si/ss:.1f}x more heterogeneous" if ss>0 else "")
print("  If ImageNet-C carries MORE between-family variance than our suite, the overstatement")
print("  documented in this paper is LARGER for the benchmark the field actually uses.")
print("  That moves the central claim from a note about our suite to one about ImageNet-C.")
print("\nIf tau is high and the cost is ~0, SDI-C is redundant and the manuscript should say so.")
print("If tau is low or the cost is material, the domain-specific suite changes the decision,")
print("which is exactly the value proposition the reviewer said was asserted but not tested.")

suite,inc,sdic,rank_sdic,rank_inc
backbone,,,,
convnext_tiny,0.6898,0.8180,3.0,3.0
resnet50,0.6041,0.7383,5.0,5.0
tf_efficientnet_b0,0.6752,0.7614,4.0,4.0
vit_base_patch16_224,0.7089,0.8331,2.0,2.0
vit_small_patch14_reg4_dinov2,0.7612,0.8494,1.0,1.0



CONCERN 6 — ranking agreement between the suites
  Kendall tau = +1.000  (p = 0.017)
  most robust on SDI-C       : vit_small_patch14_reg4_dinov2
  most robust on ImageNet-C  : vit_small_patch14_reg4_dinov2
  cost of choosing on ImageNet-C, measured on SDI-C: +0.0000 balanced accuracy


,median,min,max
suite,,,
inc,0.1372,0.1039,0.2370
sdic,0.0562,0.0157,0.1395



FAMILY-LEVEL HETEROGENEITY BY SUITE
  ImageNet-C rho median 0.137 | SDI-C rho median 0.056  -> ImageNet-C 2.4x more heterogeneous
  If ImageNet-C carries MORE between-family variance than our suite, the overstatement
  documented in this paper is LARGER for the benchmark the field actually uses.
  That moves the central claim from a note about our suite to one about ImageNet-C.

If tau is high and the cost is ~0, SDI-C is redundant and the manuscript should say so.
If tau is low or the cost is material, the domain-specific suite changes the decision,
which is exactly the value proposition the reviewer said was asserted but not tested.


In [ ]:
#@title Figures for the manuscript
fig,ax=plt.subplots(1,2,figsize=(11,3.8))
g=V[V.comparison=="C4 vs C2"].sort_values("rho")
ax[0].errorbar(g.rho,range(len(g)),
               xerr=[g.rho-g.rho_lo,g.rho_hi-g.rho],fmt="o",ms=5,capsize=3,lw=1)
ax[0].set_yticks(range(len(g))); ax[0].set_yticklabels(g.backbone,fontsize=7)
ax[0].axvline(0,ls="--",c="0.5",lw=.8); ax[0].set_xlabel("family-level ICC  ρ")
ax[0].set_title("Concern 3: ρ across backbones (C4 vs C2)",fontsize=9)

w=0.35; x=np.arange(len(rank))
ax[1].bar(x-w/2,rank["sdic"],w,label="SDI-C"); ax[1].bar(x+w/2,rank["inc"],w,label="ImageNet-C")
ax[1].set_xticks(x); ax[1].set_xticklabels(rank.index,rotation=20,ha="right",fontsize=7)
ax[1].set_ylabel("balanced accuracy under corruption")
ax[1].set_title(f"Concern 6: suite agreement, τ = {tau.statistic:+.2f}",fontsize=9)
ax[1].legend(fontsize=7,frameon=False)
plt.tight_layout(); plt.savefig(OUT/"phase1_figures.png",dpi=200,bbox_inches="tight"); plt.show()

json.dump({"concern3":{"rho_median":float(V.rho.median()),
                       "rho_range":[float(V.rho.min()),float(V.rho.max())],
                       "se_ratio_range":[float(V.se_ratio.min()),float(V.se_ratio.max())],
                       "all_intervals_exclude_zero":bool((V.rho_lo>0).all())},
           "concern6":{"kendall_tau":float(tau.statistic),"p":float(tau.pvalue),
                       "best_sdic":str(best_s),"best_imagenetc":str(best_i),
                       "cost_of_wrong_suite":float(cost)}},
          open(OUT/"phase1_summary.json","w"),indent=2)
print("wrote", OUT/"phase1_summary.json")

---

## Writing the results up

**Concern 3.** Replace every point estimate in Section 6.6 with the across-backbone range. If all
five intervals exclude zero, the sentence becomes *"family-level heterogeneity is present for
every backbone tested, with ρ between X and Y"* — a claim about the benchmark rather than about
ResNet-50, which is what the reviewer asked for.

**Concern 6.** Whichever way it comes out, report it in Section 3 where SDI-C is introduced, not
buried in the results. If Kendall's τ is high and the cost of choosing on ImageNet-C is
negligible, say plainly that a generic suite would have served, and justify SDI-C on
interpretability instead — the vignetting diagnosis, which no generic family provides. If τ is low,
that number belongs in the abstract.

**What is still not addressed.** Both experiments run on NEU-CLS. Magnetic Tile needs folds
rebuilt without the discarded pHash grouping; KolektorSDD2 needs the official ViCoS release. Until
one of those lands, the Limitations section must keep saying *one dataset*.